# Explainable Hybrid Power Quality Disturbance Classification

This notebook implements a complete, edge-ready pipeline for PQD classification featuring three primary novel contributions over traditional methods:
1. **1D Residual TCN**: Replacing expensive 2D CWT transforms with direct time-domain processing.
2. **Noise Immunity Profiling**: Utilizing a Denoising Autoencoder (DAE) to maintain robustness across severe SNR degradation.
3. **1D Grad-CAM Explainability**: Providing visual, physical interpretation of model attention directly on the voltage waveform.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1️⃣ Load XPQRS CSVs into a long-form DataFrame
DATA_DIR = "/kaggle/input/seed-power-quality-disturbance-dataset/XPQRS"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))
dfs = []
for fp in csv_paths:
    label = os.path.splitext(os.path.basename(fp))[0]
    df0 = pd.read_csv(fp)
    df_long = df0.melt(var_name="instance", value_name="amplitude")
    df_long["time_idx"] = df_long.groupby("instance").cumcount()
    df_long["label"]    = label
    dfs.append(df_long)
full_df = pd.concat(dfs, ignore_index=True)

# 2️⃣ Pivot to get X raw and y
pivot = full_df.pivot_table(
    index=["label","instance"],
    columns="time_idx",
    values="amplitude"
)
X_raw = pivot.values.astype("float32")  # shape (1700, 999)
labels = pivot.index.get_level_values("label")
le = LabelEncoder()
y = le.fit_transform(labels)            # 0..16

# 3️⃣ Train/test split & Reshape for 1D convolutions
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

X_train_1d = X_train_raw[..., np.newaxis]
X_test_1d  = X_test_raw[...,  np.newaxis]

print("Data loaded and shaped for 1D input:", X_train_1d.shape)

## Phase 1: The Denoising Autoencoder (DAE)

In [ ]:
# Simulate 5% baseline noise for DAE training
noise_factor = 0.05
X_train_noisy = X_train_1d + noise_factor * np.random.normal(size=X_train_1d.shape)
X_test_noisy  = X_test_1d  + noise_factor * np.random.normal(size=X_test_1d.shape)

# Build the encoder-decoder
inp = layers.Input((999,1))
# Encoder
x = layers.Conv1D(16, 3, padding='same', activation='relu')(inp)
x = layers.MaxPool1D(2, padding='same')(x)   
x = layers.Conv1D(8, 3, padding='same', activation='relu')(x)
encoded = layers.MaxPool1D(2, padding='same')(x) 
# Decoder
x = layers.Conv1D(8, 3, padding='same', activation='relu')(encoded)
x = layers.UpSampling1D(2)(x)                
x = layers.Conv1D(16, 3, padding='same', activation='relu')(x)
x = layers.UpSampling1D(2)(x)                
x = layers.Conv1D(1, 3, padding='same', activation='linear')(x)
decoded = layers.Cropping1D((0,1))(x)        

autoencoder = Model(inp, decoded, name="Denoising_Autoencoder")
autoencoder.compile(optimizer='adam', loss='mse')

print("Training DAE...")
history_ae = autoencoder.fit(
    X_train_noisy, X_train_1d,
    epochs=20, batch_size=32, shuffle=True,
    validation_data=(X_test_noisy, X_test_1d),
    verbose=1
)

## Phase 2: Contribution 1 - 1D Residual TCN (Stage 2 Replacement)

In [ ]:
def residual_tcn_block(x, filters, dilation_rate):
    """A single Residual block for the TCN with causal dilation."""
    conv = layers.Conv1D(filters, kernel_size=3, padding='causal', 
                         dilation_rate=dilation_rate, activation='relu')(x)
    conv = layers.Conv1D(filters, kernel_size=3, padding='causal', 
                         dilation_rate=dilation_rate)(conv)
    
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, kernel_size=1, padding='same')(x)
        
    res = layers.Add()([x, conv])
    return layers.Activation('relu')(res)

def build_1d_tcn(input_shape=(999, 1), num_classes=17):
    inp = layers.Input(shape=input_shape)
    x = inp
    
    dilation_rates = [1, 2, 4, 8, 16, 32]
    for dilation in dilation_rates:
        x = residual_tcn_block(x, filters=32, dilation_rate=dilation)
    
    # Crucial: Name this layer for Grad-CAM targeting later
    x = layers.Conv1D(64, kernel_size=3, padding='causal', activation='relu', name='target_conv_layer')(x)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    
    return Model(inp, out, name="1D_Residual_TCN")

tcn_model = build_1d_tcn(input_shape=(999, 1), num_classes=len(le.classes_))
tcn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
]

print("\nTraining 1D Residual TCN...")
history_tcn = tcn_model.fit(
    X_train_1d, y_train,
    validation_data=(X_test_1d, y_test),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## Phase 3: Contribution 2 - Noise Immunity Profiling
Evaluating the true grid readiness of the pipeline by injecting dynamic SNR conditions.

In [ ]:
def add_awgn_noise_snr(signal_array, snr_db):
    noisy_signals = np.zeros_like(signal_array)
    for i in range(signal_array.shape[0]):
        signal = signal_array[i]
        sig_power = np.mean(signal ** 2)
        snr_linear = 10 ** (snr_db / 10.0)
        noise_power = sig_power / snr_linear
        noise = np.random.normal(0, np.sqrt(noise_power), size=signal.shape)
        noisy_signals[i] = signal + noise
    return noisy_signals

snr_levels_db = [20, 25, 30, 35, 40, 45, 50]
baseline_acc = []
dae_acc = []

print("\n--- Running Noise Immunity Profile ---")
for snr in snr_levels_db:
    X_test_noisy = add_awgn_noise_snr(X_test_1d, snr)
    
    # Baseline (No DAE)
    preds_noisy = np.argmax(tcn_model.predict(X_test_noisy, verbose=0), axis=1)
    baseline_acc.append(accuracy_score(y_test, preds_noisy))
    
    # Robustness Pipeline (DAE + TCN)
    X_test_cleaned = autoencoder.predict(X_test_noisy, verbose=0)
    preds_cleaned = np.argmax(tcn_model.predict(X_test_cleaned, verbose=0), axis=1)
    dae_acc.append(accuracy_score(y_test, preds_cleaned))
    
    print(f"SNR: {snr}dB | No DAE Acc: {baseline_acc[-1]:.3f} | With DAE Acc: {dae_acc[-1]:.3f}")

plt.figure(figsize=(10, 6))
plt.plot(snr_levels_db, baseline_acc, marker='o', linestyle='dashed', color='red', label='Without DAE')
plt.plot(snr_levels_db, dae_acc, marker='s', linestyle='-', color='green', label='With DAE (Proposed)')
plt.title('Noise Immunity Profile: Accuracy vs SNR', fontsize=14)
plt.xlabel('Signal-to-Noise Ratio (dB) [Lower is Noisier]', fontsize=12)
plt.ylabel('Classification Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.gca().invert_xaxis() # Noise increases left to right
plt.show()

## Phase 4: Contribution 3 - 1D Grad-CAM Explainability
Shattering the black box by projecting model attention onto the physical waveform.

In [ ]:
def make_gradcam_heatmap_1d(inputs, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def plot_gradcam_1d(signal, heatmap, title):
    t = np.linspace(0, 20, len(signal)) 
    fig, ax = plt.subplots(figsize=(12, 4))
    
    ax.plot(t, signal, color='white', linewidth=1.5, label='Voltage Waveform', zorder=2)
    ymin, ymax = ax.get_ylim()
    extent = [t[0], t[-1], ymin, ymax]
    
    img = ax.imshow(heatmap[np.newaxis, :], cmap='jet', aspect='auto', alpha=0.8, extent=extent, zorder=1)
    ax.set_facecolor('black')
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude (p.u.)')
    plt.colorbar(img, ax=ax, label='Grad-CAM Attention (Significance)')
    plt.tight_layout()
    plt.show()

# Isolate a complex hybrid fault for the demonstration
sample_idx = 42 # Change this to explore different disturbances
test_sample = X_test_1d[sample_idx:sample_idx+1]
true_label_name = le.inverse_transform([y_test[sample_idx]])[0]

print(f"\nGenerating Explanation for: {true_label_name}")
heatmap_1d = make_gradcam_heatmap_1d(test_sample, tcn_model, 'target_conv_layer')
plot_gradcam_1d(test_sample[0].flatten(), heatmap_1d, title=f"Interpretability Analysis: {true_label_name}")